# Telemetry Dashboard Preparation

**Author**: Patricio Ortiz

In this notebook we will create the main functions of the Pipeline that will take telemetry data as input and will output the different analysis.
The input data of telemetry raw signals data is located on ../data/telemetry/silver/{client}Telemetry_Wide_With_States/
The input data of telemetry metadata signals is located on ../data/telemetry/config/{client}/signal_registry.yaml
The input data of equipments metadata is located on ../data/telemetry/config/{client}/equipment_registry.yaml

The files on that folder have the following format : {ww-yyyy}.parquet

Inside each file we have the data stored on rows with the following columns : 
unitId | timeStart | state | feature_1 | ... | feature_n

where feature_X represents one of many features inside the dataset.

## Input Data

In [1]:
import pandas as pd
import numpy as np
import yaml

# read telemetry data
def read_telemetry_data(file_path):
    """
    Reads telemetry data from a parquet file and returns a pandas DataFrame.
    
    Parameters:
    file_path (str): The path to the parquet file containing the telemetry data.
    
    Returns:
    pd.DataFrame: A DataFrame containing the telemetry data.
    """
    try:
        df = pd.read_parquet(file_path)
        return df
    except Exception as e:
        print(f"Error reading telemetry data: {e}")
        return None

# read yaml file with metadata
def read_yaml(file_path):
    """
    Reads a YAML file and returns its contents as a dictionary.
    
    Parameters:
    file_path (str): The path to the YAML file to be read.
    
    Returns:
    dict: A dictionary containing the contents of the YAML file.
    """
    try:
        with open(file_path, 'r') as file:
            data = yaml.safe_load(file)
        return data
    except Exception as e:
        print(f"Error reading YAML file: {e}")
        return None
    
# File paths
TELEMETRY_PATH = "../data/telemetry/silver/cda/Telemetry_Wide_With_States"
TELEMETRY_METADATA_PATH = "../data/telemetry/config/cda/signal_registry.yaml"
EQUIPMENT_METADATA_PATH = "../data/telemetry/config/cda/equipment_registry.yaml"

# Read telemetry data and metadata
df = read_telemetry_data(f"{TELEMETRY_PATH}")
telemetry_metadata = read_yaml(TELEMETRY_METADATA_PATH)
equipment_metadata = read_yaml(EQUIPMENT_METADATA_PATH)

In [2]:
df.columns

Index(['Fecha', 'Unit', 'Estado', 'EstadoMaquina', 'EstadoCarga', 'GPSLat',
       'GPSLon', 'GPSElevation', 'AmbAirTemp', 'AtmosPres', 'BoostPres',
       'CompInPres1', 'CompInPres2', 'LckupSlip', 'ThrottlPos', 'TrboInPres',
       'TrboOutPres', 'TrnSlip', 'TrnGear', 'GearSelect', 'AirFltr',
       'CnkcasePres', 'DiffLubePres', 'DiffTemp', 'EngCoolTemp', 'EngOilFltr',
       'EngOilPres', 'EngSpd', 'GroundSpd', 'LtExhTemp', 'LtFBrkTemp',
       'LtRBrkTemp', 'Payload', 'RAftrclrTemp', 'RtExhTemp', 'RtFBrkTemp',
       'RtLtExhTemp', 'RtRBrkTemp', 'StrgOilTemp', 'TCOutTemp', 'TrnLubeTemp'],
      dtype='object')

In [3]:
telemetry_metadata['signals']

[{'name': 'EngCoolTemp',
  'display_name': 'Engine Coolant Temperature',
  'system': 'Engine',
  'subsystem': 'Cooling',
  'unit': 'Â°C',
  'risk_direction': 'high',
  'threshold_compute': True,
  'physical_min': 0.0,
  'physical_max': 150.0,
  'criticality': 3,
  'description': 'Engine coolant temperature - critical for thermal management'},
 {'name': 'EngOilPres',
  'display_name': 'Engine Oil Pressure',
  'system': 'Engine',
  'subsystem': 'Lubrication',
  'unit': 'kPa',
  'risk_direction': 'low',
  'threshold_compute': True,
  'physical_min': 0.0,
  'physical_max': 800.0,
  'criticality': 3,
  'description': 'Engine oil pressure - critical for engine protection'},
 {'name': 'EngOilFltr',
  'display_name': 'Engine Oil Filter',
  'system': 'Engine',
  'subsystem': 'Lubrication',
  'unit': 'kPa',
  'risk_direction': 'high',
  'threshold_compute': False,
  'physical_min': 0.0,
  'physical_max': 500.0,
  'criticality': 2,
  'description': 'Engine oil filter differential pressure'},
 {'n

In [4]:
equipment_metadata['equipments']

[{'name': 'T_09',
  'brand': 'Caterpillar',
  'model': '789C',
  'has_silencer': False},
 {'name': 'T_10',
  'brand': 'Caterpillar',
  'model': '789C',
  'has_silencer': False},
 {'name': 'T_11',
  'brand': 'Caterpillar',
  'model': '789C',
  'has_silencer': False},
 {'name': 'T_12',
  'brand': 'Caterpillar',
  'model': '789C',
  'has_silencer': False},
 {'name': 'T_13',
  'brand': 'Caterpillar',
  'model': '789C',
  'has_silencer': False},
 {'name': 'T_14',
  'brand': 'Caterpillar',
  'model': '789C',
  'has_silencer': False},
 {'name': 'T_15',
  'brand': 'Caterpillar',
  'model': '789C',
  'has_silencer': True},
 {'name': 'T_16',
  'brand': 'Caterpillar',
  'model': '789C',
  'has_silencer': False},
 {'name': 'T_17',
  'brand': 'Caterpillar',
  'model': '789C',
  'has_silencer': False},
 {'name': 'T_18',
  'brand': 'Caterpillar',
  'model': '789D',
  'has_silencer': False},
 {'name': 'T_24',
  'brand': 'Caterpillar',
  'model': '789D',
  'has_silencer': False}]

## Telemetry Analysis

Here we perform the analysis over raw data to get a deep-dive view of the telemetry data.

Analysis to perform include:
* Deviation Analysis : Compare value against thresholds to categorize in (Normal - Alerta - Anormal - Critico). The ranges are dependant on percentiles and analyzed based on the "risk_direction" parameter of metadata. (Example : If risk_direction==high => we use as thresholds P95 - P98 - P99

### Deviation Analysis

First, we start with a deviation analysis. In this section we compute the limits of the features based on states. 

#### Computation Details

**Limits Calculation:**
- Limits are computed per each `model_specification`
- Values are taken from `equipment_metadata` focusing on the unique `model_{with_silencer}`

**Percentiles Computed:**
- We compute the following percentiles: **P1, P2, P5, P10, P25, P50, P75, P90, P95, P98, P99**

**Scope:**
- Computed for every combination of:
  - Model
  - State
  - Feature (where `threshold_compute == true` in `signal_registry`)

#### Implementation Notes

The thresholds will be used to categorize values into:
- Normal
- Alerta
- Anormal
- Crítico

The ranges are dependent on percentiles and analyzed based on the `risk_direction` parameter from metadata.

In [5]:
UNIT_COLNAME = 'Unit'
STATE_COLNAME = 'Estado'
TIME_COLNAME = 'Fecha'

# Compute model_specification column based on equipment metadata and unit identifier column
def compute_model_specification(
    df_in:pd.DataFrame,
    equipment_metadata:dict,
    colname:str=UNIT_COLNAME
    ) -> pd.DataFrame:
    '''
    Compute the model_specification column based on equipment metadata and unit identifier column.

    Parameters:
        - df_in (pd.DataFrame): Input DataFrame containing telemetry data.
        - equipment_metadata (dict): Dictionary containing equipment metadata.
        - colname (str): The name of the column containing unit identifiers. Default is 'Unit'.

    Returns:
        - pd.DataFrame: DataFrame with an additional 'model_specification' column.
    '''
    df_out = df_in.copy()
    # Create a mapping of unit_identifier to model_with_silencer
    unit_to_model = {}
    for equipment in equipment_metadata['equipments']:
        model_specification_string = f'{equipment["model"]}_with_silencer' if equipment.get('has_silencer', False) else equipment['model']
        unit_to_model[equipment['name']] = model_specification_string
    # Map the unit identifiers in the telemetry data to model specifications
    df_out.loc[:, 'model_specification'] = df_out[colname].map(unit_to_model)
    df_out = df_out.dropna(subset=['model_specification'])
    return df_out

# Compute limits for all features that require threshold computation
def compute_limits(
    df_in:pd.DataFrame,
    telemetry_metadata:dict,
    equipment_metadata:dict
    ) -> dict:
    '''
    Compute limits for all features that require threshold computation.
    Parameters:
        - df_in (pd.DataFrame): Input DataFrame containing telemetry data with model specifications.
        - telemetry_metadata (dict): Dictionary containing telemetry metadata.
        - equipment_metadata (dict): Dictionary containing equipment metadata.
    Returns:
        - dict: A dictionary containing the computed limits for each feature.
            format : {model_specification: {feature: {state: {percentiles}}}}
    '''
    
    limits = {}
    
    valid_pairs = df_in[['model_specification', 'Estado']].drop_duplicates().dropna(how='any')
    valid_pairs = [tuple(x) for x in valid_pairs.to_numpy() if 'nan' not in x]
    
    features_to_compute = [signal['name'] for signal in telemetry_metadata['signals'] if signal.get('threshold_compute', False)]
    
    for model_specification, state in valid_pairs:
        if model_specification not in limits:
            limits[model_specification] = {}
        
        for feature in features_to_compute:
            
            feature_values = df_in[(df_in['model_specification'] == model_specification) & (df_in['Estado'] == state)][feature].dropna()
            if feature_values.unique().size > 9: # We need at least 10 unique values to compute percentiles
                
                if feature not in limits[model_specification]:
                    limits[model_specification][feature] = {}
                if state not in limits[model_specification][feature]:
                    limits[model_specification][feature][state] = {}
                
                p = np.percentile(feature_values, [1, 2, 5, 10, 25, 50, 75, 90, 95, 98, 99])
                limits[model_specification][feature][state] = {
                    'P1': np.round(p[0],1),
                    'P2': np.round(p[1],1),
                    'P5': np.round(p[2],1),
                    'P10': np.round(p[3],1),
                    'P25': np.round(p[4],1),
                    'P50': np.round(p[5],1),
                    'P75': np.round(p[6],1),
                    'P90': np.round(p[7],1),
                    'P95': np.round(p[8],1),
                    'P98': np.round(p[9],1),
                    'P99': np.round(p[10],1)
                }
            
    return limits

# Select the limits for a specific model, feature and state
def select_limits(limits_per_model:dict, feature:str, risk_direction:str) -> dict:
    """
    Select the limits for a specific model, feature and risk direction.
    Limits are :
        - alert_threshold : P5/P95 (depending on the risk direction)
        - anormal_threshold : P2/P98 (depending on the risk direction)
        - critical_threshold : P1/P99 (depending on the risk direction)
    
    Parameters:
        - limits_per_model (dict): A dictionary containing the limits for each model and feature.
        - feature (str): The feature for which to select the limits.
        - risk_direction (str): The risk direction ('high', 'low', or 'both').
        
    Returns:
        - dict: A dictionary containing the selected limits for the specified feature and risk direction.
            Since we have risk = 'both' as an option the values of the limits will be a list, hence the format is Dict[list].
            
    Raises:
        - ValueError: If the risk direction is invalid or if the feature is not found in the limits for the model.
    """
    if risk_direction not in ['high', 'low', 'both']:
        raise ValueError("Invalid risk direction. Must be 'high', 'low' or 'both'.")
    else:
        if feature not in limits_per_model.keys():
            raise ValueError(f"Feature '{feature}' not found in limits for the model.")
        else:
            if risk_direction == 'high':
                return {state : {
                    'alert_threshold': [limits_per_model[feature][state]['P95']],
                    'anormal_threshold': [limits_per_model[feature][state]['P98']],
                    'critical_threshold': [limits_per_model[feature][state]['P99']]
                    } for state in limits_per_model[feature].keys()
                }
            elif risk_direction == 'low':
                return {state : {
                    'alert_threshold': [limits_per_model[feature][state]['P5']],
                    'anormal_threshold': [limits_per_model[feature][state]['P2']],
                    'critical_threshold': [limits_per_model[feature][state]['P1']]
                    } for state in limits_per_model[feature].keys()
                }
            else: # risk_direction == 'both'
                return {state : {
                    'alert_threshold': [limits_per_model[feature][state]['P5'], limits_per_model[feature][state]['P95']],
                    'anormal_threshold': [limits_per_model[feature][state]['P2'], limits_per_model[feature][state]['P98']],
                    'critical_threshold': [limits_per_model[feature][state]['P1'], limits_per_model[feature][state]['P99']]
                    } for state in limits_per_model[feature].keys()
                }     

# Compare limits for a specific unit, feature and time-window
def compare_limits_unit_feature_time(
    df_in:pd.DataFrame,
    feature_name:str,
    thresholds:dict,
    risk_direction:str
) -> pd.DataFrame:
    '''
    Compare limits for a specific model, feature and time-window.
    A time-window data arrives with many units (therefore many models) and many states.
    This function will compare the values of the feature with the thresholds, and will return a DataFrame with the comparison results.
    
    Parameters:
        - df_in (pd.DataFrame): Input DataFrame containing telemetry data for a specific model, feature and time-window.
        - feature_name (str): The name of the feature for which to compare the limits.
        - thresholds (dict): A dictionary containing the thresholds for the specific model, feature and state. (It has state as main key and inside each state we have 3 thresholds : alert_threshold, anormal_threshold and critical_threshold).
        - risk_direction (str): The risk direction ('high', 'low', or 'both').
            - If 'high' we apply a >= comparison with the thresholds, and the risk levels are 'normal', 'alert', 'anormal' and 'critical'.
            - If 'low' we apply a <= comparison with the thresholds, and the risk levels are 'normal', 'alert', 'anormal' and 'critical'.
            - If 'both' we apply a between comparison with the thresholds, and the risk levels are 'normal', 'alert', 'anormal' and 'critical'.
                   
    Returns:
        - pd.DataFrame: A DataFrame containing the comparison results, with an additional column 'risk_level' indicating the risk level based on the thresholds.
    '''
    if risk_direction not in ['high', 'low', 'both']:
        raise ValueError("Invalid risk direction. Must be 'high', 'low' or 'both'.")
    
    df_out = df_in.copy()
    def categorize_value(row):
        state = row[STATE_COLNAME]
        value = row[feature_name]
        
        # Check if value is null
        if pd.isna(value):
            return 'unknown'
        
        # Check if state exists in thresholds
        if state not in thresholds:
            return 'unknown'
        
        state_thresholds = thresholds[state]
        alert_threshold = state_thresholds['alert_threshold']
        anormal_threshold = state_thresholds['anormal_threshold']
        critical_threshold = state_thresholds['critical_threshold']
        
        if risk_direction == 'high':
            # For high risk, higher values are worse
            # Thresholds are single values in a list
            if value < alert_threshold[0]:
                return 'normal'
            elif value < anormal_threshold[0]:
                return 'alert'
            elif value < critical_threshold[0]:
                return 'anormal'
            else:
                return 'critical'
        
        elif risk_direction == 'low':
            # For low risk, lower values are worse
            if value > alert_threshold[0]:
                return 'normal'
            elif value > anormal_threshold[0]:
                return 'alert'
            elif value > critical_threshold[0]:
                return 'anormal'
            else:
                return 'critical'
        
        else:  # risk_direction == 'both'
            # For both, values outside the range are worse
            # Each threshold has two values: [lower, upper]
            lower_critical = critical_threshold[0]
            upper_critical = critical_threshold[1]
            lower_anormal = anormal_threshold[0]
            upper_anormal = anormal_threshold[1]
            lower_alert = alert_threshold[0]
            upper_alert = alert_threshold[1]
            
            # Check if outside critical range
            if value <= lower_critical or value >= upper_critical:
                return 'critical'
            # Check if outside anormal range
            elif value <= lower_anormal or value >= upper_anormal:
                return 'anormal'
            # Check if outside alert range
            elif value <= lower_alert or value >= upper_alert:
                return 'alert'
            else:
                return 'normal'
    
    df_out[f'risk_level_{feature_name}'] = df_out.apply(categorize_value, axis=1)
    df_out.set_index([UNIT_COLNAME, TIME_COLNAME], inplace=True)
    
    return df_out[[f'risk_level_{feature_name}']]

# Compare limits for a specific time-window
def compare_limits(
    df_in:pd.DataFrame,
    limits:dict,
    telemetry_metadata:dict,
    ) -> pd.DataFrame:
    '''
    Compare limits for a specific time-window.
    A time-window data arrives with many units (therefore many models) many states and many features.
    This function will compare the values of the features with the limits computed for each model and state, and will return a DataFrame with the comparison results.
    '''    
    labels = []
    for model in df_in['model_specification'].unique():
        labeled_features = []
        for feature in telemetry_metadata['signals']:
            feature_name = feature['name']
            if feature_name in limits[model].keys():
                df_subset = df_in[(df_in['model_specification'] == model)][[UNIT_COLNAME, TIME_COLNAME, STATE_COLNAME, feature_name]]
                selected_limits = select_limits(limits[model], feature_name, feature['risk_direction'])
                
                df_labeled = compare_limits_unit_feature_time(df_subset, feature_name, selected_limits, feature['risk_direction'])
                labeled_features.append(df_labeled)
        labeled_model = pd.concat(labeled_features, axis=1)
        labels.append(labeled_model)
    df_labeled = pd.concat(labels, axis=0)
    df_out = pd.concat([df_in.set_index([UNIT_COLNAME, TIME_COLNAME]), df_labeled], axis=1)
    return df_out

In [16]:
min_date = df[TIME_COLNAME].min()
threshold_date = min_date + pd.Timedelta(weeks=12)
df_aux = compute_model_specification(df[df[TIME_COLNAME] <= threshold_date], equipment_metadata)
l = compute_limits(df_aux, telemetry_metadata, equipment_metadata)
r = compare_limits(df_aux, l, telemetry_metadata)

### Event Analysis

Then, we apply an event analysis. In this section we compute additional analysis over the deviation data to identify temporal patterns.

#### Overview

Events are defined as **periods with consecutive non-normal reads** that may indicate operational issues or anomalies requiring attention.

#### Computation Details

**Data Source:**
- Values are extracted from `risk_level_{feature}` columns (output from Deviation Analysis)
- Analysis identifies temporal patterns for each unit

**Event Classification by Duration:**
- 🔸 **Spike**: Duration < 5 minutes
- 🟡 **Anomaly**: Duration < 30 minutes  
- 🔴 **Warning**: Duration ≥ 30 minutes

**Scope:**
- Computed for every combination of:
  - Unit (individual equipment)
  - Feature (telemetry signal)
- **No gaps allowed**: Events must have consecutive non-normal readings

#### Implementation Approaches

We implement two parallel modeling approaches to capture different aspects of event severity:

##### Approach 1: Binary Non-Normal Model
- **Grouping Logic**: All non-normal readings are treated equally
- **Event Duration**: Measured in **minutes**
- **Classification Thresholds**:
  - Spike: < 5 minutes
  - Anomaly: < 30 minutes
  - Warning: ≥ 30 minutes

##### Approach 2: Weighted Severity Model
- **Grouping Logic**: Risk levels are weighted by severity
- **Point System**:
  - Alert → **1 point/minute**
  - Anormal → **3 points/minute**
  - Critical → **5 points/minute**
- **Event Severity**: Measured in **accumulated points** (not minutes)
- **Classification Thresholds**:
  - Spike: < 10 points
  - Anomaly: < 30 points
  - Warning: ≥ 30 points

**Example**: A 6-minute event with 2 minutes of Alert + 2 minutes of Anormal + 2 minutes of Critical would score: `(2×1) + (2×3) + (2×5) = 20 points` → classified as **Anomaly**

In [17]:
# Event Analysis Implementation
def identify_events_unit_feature(
    df_in: pd.DataFrame,
    unit: str,
    feature: str,
    time_col: str = TIME_COLNAME
) -> pd.DataFrame:
    '''
    Identify consecutive non-normal events for a specific unit and feature.
    
    Parameters:
        - df_in (pd.DataFrame): Input DataFrame with risk_level columns (indexed by Unit and Time)
        - unit (str): The unit identifier to analyze
        - feature (str): The feature name to analyze
        - time_col (str): The name of the time column
        
    Returns:
        - pd.DataFrame: DataFrame with event groups, one row per timestamp in an event
    '''
    risk_col = f'risk_level_{feature}'
    
    # Filter for specific unit and get risk levels
    if isinstance(df_in.index, pd.MultiIndex):
        unit_data = df_in.loc[unit].copy()
    else:
        unit_data = df_in[df_in[UNIT_COLNAME] == unit].copy()
    
    if risk_col not in unit_data.columns:
        return pd.DataFrame()
    
    # Sort by time
    unit_data = unit_data.sort_index() if isinstance(unit_data.index, pd.DatetimeIndex) else unit_data.sort_values(time_col)
    
    # Create binary indicator for non-normal readings
    unit_data['is_non_normal'] = ~unit_data[risk_col].isin(['normal', 'unknown'])
    
    # Create event groups (consecutive non-normal readings)
    # Group changes when is_non_normal changes or when there's a time gap
    unit_data['event_group'] = (
        (unit_data['is_non_normal'] != unit_data['is_non_normal'].shift()) |
        (unit_data['is_non_normal'] & (unit_data.index.to_series().diff() > pd.Timedelta(minutes=1)))
    ).cumsum()
    
    # Filter only non-normal events
    events = unit_data[unit_data['is_non_normal']].copy()
    events['unit'] = unit
    events['feature'] = feature
    
    return events[[risk_col, 'event_group', 'unit', 'feature']]


def calculate_binary_event_metrics(events_df: pd.DataFrame, feature: str) -> pd.DataFrame:
    '''
    Calculate event metrics using Binary Non-Normal Model (Approach 1).
    
    Parameters:
        - events_df (pd.DataFrame): DataFrame with identified events
        - feature (str): The feature name
        
    Returns:
        - pd.DataFrame: Event summary with duration-based classification
    '''
    if events_df.empty:
        return pd.DataFrame()
    
    risk_col = f'risk_level_{feature}'
    
    # Group by event_group and calculate metrics
    event_summary = events_df.groupby(['unit', 'feature', 'event_group']).agg(
        start_time=('event_group', lambda x: x.index.min()),
        end_time=('event_group', lambda x: x.index.max()),
        duration_minutes=('event_group', lambda x: len(x)),
        max_severity=(risk_col, lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0])
    ).reset_index()
    
    # Classify based on duration
    def classify_binary(duration):
        if duration < 5:
            return 'spike'
        elif duration < 30:
            return 'anomaly'
        else:
            return 'warning'
    
    event_summary['event_type_binary'] = event_summary['duration_minutes'].apply(classify_binary)
    
    return event_summary


def calculate_weighted_event_metrics(events_df: pd.DataFrame, feature: str) -> pd.DataFrame:
    '''
    Calculate event metrics using Weighted Severity Model (Approach 2).
    
    Parameters:
        - events_df (pd.DataFrame): DataFrame with identified events
        - feature (str): The feature name
        
    Returns:
        - pd.DataFrame: Event summary with severity-based classification
    '''
    if events_df.empty:
        return pd.DataFrame()
    
    risk_col = f'risk_level_{feature}'
    
    # Define severity weights
    severity_weights = {
        'alert': 1,
        'anormal': 3,
        'critical': 5
    }
    
    # Calculate weighted points
    events_df['severity_points'] = events_df[risk_col].map(severity_weights).fillna(0)
    
    # Group by event_group and calculate metrics
    event_summary = events_df.groupby(['unit', 'feature', 'event_group']).agg(
        start_time=('event_group', lambda x: x.index.min()),
        end_time=('event_group', lambda x: x.index.max()),
        total_points=('severity_points', 'sum'),
        duration_minutes=('event_group', lambda x: len(x)),
        alert_minutes=(risk_col, lambda x: (x == 'alert').sum()),
        anormal_minutes=(risk_col, lambda x: (x == 'anormal').sum()),
        critical_minutes=(risk_col, lambda x: (x == 'critical').sum())
    ).reset_index()
    
    # Classify based on severity points
    def classify_weighted(points):
        if points < 10:
            return 'spike'
        elif points < 30:
            return 'anomaly'
        else:
            return 'warning'
    
    event_summary['event_type_weighted'] = event_summary['total_points'].apply(classify_weighted)
    
    return event_summary


def identify_normal_periods_unit_feature(
    df_in: pd.DataFrame,
    unit: str,
    feature: str,
    time_col: str = TIME_COLNAME
) -> pd.DataFrame:
    '''
    Identify consecutive normal operating periods for a specific unit and feature.
    
    Parameters:
        - df_in (pd.DataFrame): Input DataFrame with risk_level columns (indexed by Unit and Time)
        - unit (str): The unit identifier to analyze
        - feature (str): The feature name to analyze
        - time_col (str): The name of the time column
        
    Returns:
        - pd.DataFrame: DataFrame with normal period groups, one row per timestamp in a normal period
    '''
    risk_col = f'risk_level_{feature}'
    
    # Filter for specific unit and get risk levels
    if isinstance(df_in.index, pd.MultiIndex):
        unit_data = df_in.loc[unit].copy()
    else:
        unit_data = df_in[df_in[UNIT_COLNAME] == unit].copy()
    
    if risk_col not in unit_data.columns:
        return pd.DataFrame()
    
    # Sort by time
    unit_data = unit_data.sort_index() if isinstance(unit_data.index, pd.DatetimeIndex) else unit_data.sort_values(time_col)
    
    # Create binary indicator for normal readings
    unit_data['is_normal'] = unit_data[risk_col] == 'normal'
    
    # Create period groups (consecutive normal readings)
    # Group changes when is_normal changes or when there's a time gap
    unit_data['period_group'] = (
        (unit_data['is_normal'] != unit_data['is_normal'].shift()) |
        (unit_data['is_normal'] & (unit_data.index.to_series().diff() > pd.Timedelta(minutes=1)))
    ).cumsum()
    
    # Filter only normal periods
    normal_periods = unit_data[unit_data['is_normal']].copy()
    normal_periods['unit'] = unit
    normal_periods['feature'] = feature
    
    return normal_periods[[risk_col, 'period_group', 'unit', 'feature']]


def calculate_normal_period_metrics(periods_df: pd.DataFrame, feature: str) -> pd.DataFrame:
    '''
    Calculate metrics for normal operating periods.
    
    Parameters:
        - periods_df (pd.DataFrame): DataFrame with identified normal periods
        - feature (str): The feature name
        
    Returns:
        - pd.DataFrame: Period summary with duration and timing information
    '''
    if periods_df.empty:
        return pd.DataFrame()
    
    # Group by period_group and calculate metrics
    period_summary = periods_df.groupby(['unit', 'feature', 'period_group']).agg(
        start_time=('period_group', lambda x: x.index.min()),
        end_time=('period_group', lambda x: x.index.max()),
        duration_minutes=('period_group', lambda x: len(x))
    ).reset_index()
    
    return period_summary


def analyze_events(
    df_in: pd.DataFrame,
    telemetry_metadata: dict,
    approach: str = 'both',
    include_normal_periods: bool = False
) -> dict:
    '''
    Analyze events across all units and features using specified approach(es).
    
    Parameters:
        - df_in (pd.DataFrame): Input DataFrame with risk_level columns (output from compare_limits)
        - telemetry_metadata (dict): Dictionary containing telemetry metadata
        - approach (str): Analysis approach - 'binary', 'weighted', or 'both' (default)
        - include_normal_periods (bool): Whether to include analysis of normal operating periods (default: False)
        
    Returns:
        - dict: Dictionary containing event analysis results
            - 'binary': DataFrame with binary model results (if approach is 'binary' or 'both')
            - 'weighted': DataFrame with weighted model results (if approach is 'weighted' or 'both')
            - 'normal_periods': DataFrame with normal period analysis (if include_normal_periods is True)
    '''
    results = {}
    
    # Get all features that have risk_level columns
    risk_level_cols = [col for col in df_in.columns if col.startswith('risk_level_')]
    features = [col.replace('risk_level_', '') for col in risk_level_cols]
    
    # Get all units
    if isinstance(df_in.index, pd.MultiIndex):
        units = df_in.index.get_level_values(0).unique()
    else:
        units = df_in[UNIT_COLNAME].unique()
    
    all_binary_events = []
    all_weighted_events = []
    all_normal_periods = []
    
    # Process each unit-feature combination
    for unit in units:
        for feature in features:
            # Identify events
            events = identify_events_unit_feature(df_in, unit, feature)
            
            if not events.empty:
                # Calculate metrics based on approach
                if approach in ['binary', 'both']:
                    binary_metrics = calculate_binary_event_metrics(events, feature)
                    if not binary_metrics.empty:
                        all_binary_events.append(binary_metrics)
                
                if approach in ['weighted', 'both']:
                    weighted_metrics = calculate_weighted_event_metrics(events, feature)
                    if not weighted_metrics.empty:
                        all_weighted_events.append(weighted_metrics)
            
            # Identify normal periods if requested
            if include_normal_periods:
                normal_periods = identify_normal_periods_unit_feature(df_in, unit, feature)
                if not normal_periods.empty:
                    normal_metrics = calculate_normal_period_metrics(normal_periods, feature)
                    if not normal_metrics.empty:
                        all_normal_periods.append(normal_metrics)
    
    # Combine results
    if approach in ['binary', 'both'] and all_binary_events:
        results['binary'] = pd.concat(all_binary_events, ignore_index=True)
    
    if approach in ['weighted', 'both'] and all_weighted_events:
        results['weighted'] = pd.concat(all_weighted_events, ignore_index=True)
    
    if include_normal_periods and all_normal_periods:
        results['normal_periods'] = pd.concat(all_normal_periods, ignore_index=True)
    
    return results

In [18]:
# Test Event Analysis
events_results = analyze_events(r, telemetry_metadata, approach='both', include_normal_periods=True)

# Quick overview
print("📊 Event Analysis Summary")
print(f"Total Events (Binary): {len(events_results['binary'])}")
print(f"  - Spikes: {(events_results['binary']['event_type_binary'] == 'spike').sum()}")
print(f"  - Anomalies: {(events_results['binary']['event_type_binary'] == 'anomaly').sum()}")
print(f"  - Warnings: {(events_results['binary']['event_type_binary'] == 'warning').sum()}")

print(f"\nTotal Events (Weighted): {len(events_results['weighted'])}")
print(f"  - Spikes: {(events_results['weighted']['event_type_weighted'] == 'spike').sum()}")
print(f"  - Anomalies: {(events_results['weighted']['event_type_weighted'] == 'anomaly').sum()}")
print(f"  - Warnings: {(events_results['weighted']['event_type_weighted'] == 'warning').sum()}")

print(f"\n✅ Normal Periods: {len(events_results['normal_periods'])}")
print(f"  - Avg Duration: {events_results['normal_periods']['duration_minutes'].mean():.1f} min")
print(f"  - Max Duration: {events_results['normal_periods']['duration_minutes'].max():.0f} min")

📊 Event Analysis Summary
Total Events (Binary): 333194
  - Spikes: 255458
  - Anomalies: 66813
  - Warnings: 10923

Total Events (Weighted): 333194
  - Spikes: 242767
  - Anomalies: 71547
  - Warnings: 18880

✅ Normal Periods: 445209
  - Avg Duration: 34.7 min
  - Max Duration: 1298 min


In [19]:
# study distribution on units
FEAT_EXAMPLE_COL = 'EngCoolTemp'

# Filter results for the selected feature
binary_feature = events_results['binary'][events_results['binary']['feature'] == FEAT_EXAMPLE_COL]
weighted_feature = events_results['weighted'][events_results['weighted']['feature'] == FEAT_EXAMPLE_COL]
normal_feature = events_results['normal_periods'][events_results['normal_periods']['feature'] == FEAT_EXAMPLE_COL]

print(f"📊 Distribution Analysis for: {FEAT_EXAMPLE_COL}")
print("="*60)

# Binary events per unit
print("\n🔸 Binary Events by Unit:")
unit_binary_counts = binary_feature.groupby('unit').agg({
    'event_group': 'count',
    'duration_minutes': ['mean', 'sum'],
    'event_type_binary': lambda x: x.mode()[0] if not x.mode().empty else 'N/A'
}).round(1)
unit_binary_counts.columns = ['event_count', 'avg_duration_min', 'total_duration_min', 'most_common_type']
print(unit_binary_counts.sort_values('event_count', ascending=False))

# Weighted events per unit
print("\n⚖️ Weighted Events by Unit:")
unit_weighted_counts = weighted_feature.groupby('unit').agg({
    'event_group': 'count',
    'total_points': ['mean', 'sum'],
    'event_type_weighted': lambda x: x.mode()[0] if not x.mode().empty else 'N/A'
}).round(1)
unit_weighted_counts.columns = ['event_count', 'avg_points', 'total_points', 'most_common_type']
print(unit_weighted_counts.sort_values('total_points', ascending=False))

# Normal periods per unit
print("\n✅ Normal Periods by Unit:")
unit_normal_counts = normal_feature.groupby('unit').agg({
    'period_group': 'count',
    'duration_minutes': ['mean', 'sum', 'max']
}).round(1)
unit_normal_counts.columns = ['period_count', 'avg_duration_min', 'total_normal_min', 'max_normal_period_min']
print(unit_normal_counts.sort_values('total_normal_min', ascending=False))

📊 Distribution Analysis for: EngCoolTemp

🔸 Binary Events by Unit:
      event_count  avg_duration_min  total_duration_min most_common_type
unit                                                                    
T_18         1677               3.4                5719            spike
T_11         1630               4.6                7496            spike
T_24         1419               2.6                3699            spike
T_12          788               6.0                4748          anomaly
T_16          691               6.0                4141          anomaly
T_14          647               5.4                3519          anomaly
T_10          611               7.8                4736          anomaly
T_13          608               5.3                3229          anomaly
T_15          584               7.6                4410          anomaly
T_17          560               5.3                2965          anomaly

⚖️ Weighted Events by Unit:
      event_count  avg_point

### Trend Analysis

Then, we apply trend analysis to identify significant changes in feature behavior over time.

#### Overview

We analyze whether features exhibit statistically significant trends that may indicate progressive degradation, improvement, or drift in equipment performance over time.

#### Computation Details

**Data Preprocessing:**
- Apply a **30-minute rolling mean** to smooth out short-term fluctuations
- Focus on the smoothed trend rather than momentary variations

**Analysis Windows:**
We evaluate trends across three time periods to capture different temporal patterns:
- **4 weeks**: Short-term trends (recent changes)
- **8 weeks**: Medium-term trends (seasonal patterns)
- **12 weeks**: Long-term trends (degradation patterns)

**Statistical Method:**
- Fit a **linear regression model** for each window:
  - Independent variable (X): Time
  - Dependent variable (Y): 30-minute rolling mean
- Evaluate **statistical significance** (p-value < 0.05)
- Calculate **slope** (rate of change per unit time)
- Compute **R² score** (goodness of fit)

**Scope:**
- Computed for every combination of:
  - Unit (individual equipment)
  - Feature (telemetry signal)
  - Time window (4, 8, 12 weeks)

#### Implementation Approach

We implement linear regression analysis to determine if observed trends are statistically meaningful and warrant user attention.

**Significance Criteria:**
- **Statistically significant**: p-value < 0.05
- **Practically significant**: Slope magnitude exceeds threshold based on feature characteristics
- **Reliable fit**: R² > 0.5 (trend explains >50% of variance)

**Output Interpretation:**
- **Positive slope + high risk direction**: Feature values increasing → potential degradation
- **Negative slope + low risk direction**: Feature values decreasing → potential degradation
- **Both directions**: Values drifting away from normal range → potential issues

#### Practical Application

Trend analysis helps identify:
- 📈 **Progressive degradation**: Gradual decline in performance over weeks
- 🔄 **Seasonal patterns**: Recurring changes in operating conditions
- ⚠️ **Early warning signs**: Subtle changes before threshold violations
- 📊 **Maintenance effectiveness**: Performance improvement post-maintenance

**Example**: Engine coolant temperature showing a statistically significant upward trend of +0.5°C/week over 12 weeks suggests potential cooling system degradation, even if current values remain within normal limits.

In [28]:
# Trend Analysis Implementation
from sklearn.linear_model import LinearRegression
from scipy import stats

R2_THRESHOLD = 0.3
P_VALUE_THRESHOLD = 0.05

def analyze_trend_unit_feature(
    df_in: pd.DataFrame,
    unit: str,
    feature: str,
    window_weeks: int = 4,
    rolling_window_minutes: int = 30,
    time_col: str = TIME_COLNAME
) -> dict:
    '''
    Analyze trend for a specific unit-feature combination over a time window.
    
    Parameters:
        - df_in (pd.DataFrame): Input DataFrame with telemetry data (indexed by Unit and Time)
        - unit (str): The unit identifier to analyze
        - feature (str): The feature name to analyze
        - window_weeks (int): Number of weeks to analyze (default: 4)
        - rolling_window_minutes (int): Rolling mean window size in minutes (default: 30)
        - time_col (str): The name of the time column
        
    Returns:
        - dict: Dictionary with trend analysis results or None if insufficient data
    '''
    
    # Filter for specific unit
    if isinstance(df_in.index, pd.MultiIndex):
        unit_data = df_in.loc[unit].copy()
    else:
        unit_data = df_in[df_in[UNIT_COLNAME] == unit].copy()
    
    if feature not in unit_data.columns:
        return None
    
    # Sort by time
    unit_data = unit_data.sort_index() if isinstance(unit_data.index, pd.DatetimeIndex) else unit_data.sort_values(time_col)
    
    # Filter data for the specified time window
    if isinstance(unit_data.index, pd.DatetimeIndex):
        max_time = unit_data.index.max()
        min_time = max_time - pd.Timedelta(weeks=window_weeks)
        window_data = unit_data[unit_data.index >= min_time]
    else:
        max_time = unit_data[time_col].max()
        min_time = max_time - pd.Timedelta(weeks=window_weeks)
        window_data = unit_data[unit_data[time_col] >= min_time].copy()
        window_data = window_data.set_index(time_col)
    
    # Apply rolling mean
    feature_series = window_data[feature].dropna()
    
    if len(feature_series) < rolling_window_minutes * 2:  # Need at least 2x rolling window size
        return None
    
    smoothed = feature_series.rolling(window=rolling_window_minutes, min_periods=1).mean()
    
    # Prepare data for regression
    valid_data = smoothed.dropna()
    if len(valid_data) < 10:  # Need minimum data points
        return None
    
    # Convert timestamps to numeric (hours since start)
    X = (valid_data.index - valid_data.index[0]).total_seconds() / 3600  # hours
    X = X.values.reshape(-1, 1)
    y = valid_data.values
    
    # Fit linear regression
    model = LinearRegression()
    model.fit(X, y)
    
    # Calculate statistics
    y_pred = model.predict(X)
    slope = model.coef_[0]
    intercept = model.intercept_
    r2 = model.score(X, y)
    
    # Calculate p-value
    n = len(X)
    residuals = y - y_pred
    mse = np.sum(residuals**2) / (n - 2)
    se_slope = np.sqrt(mse / np.sum((X - X.mean())**2))
    t_stat = slope / se_slope
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), n - 2))
    
    return {
        'unit': unit,
        'feature': feature,
        'window_weeks': window_weeks,
        'slope': slope,
        'slope_per_day': slope * 24,  # Convert from per-hour to per-day
        'intercept': intercept,
        'r2': r2,
        'p_value': p_value,
        'is_significant': p_value < P_VALUE_THRESHOLD,
        'is_good_fit': r2 > R2_THRESHOLD,
        'data_points': n,
        'start_time': valid_data.index.min(),
        'end_time': valid_data.index.max()
    }


def analyze_trends(
    df_in: pd.DataFrame,
    telemetry_metadata: dict,
    window_weeks_list: list = [4, 8, 12],
    rolling_window_minutes: int = 30
) -> pd.DataFrame:
    '''
    Analyze trends across all units and features for multiple time windows.
    
    Parameters:
        - df_in (pd.DataFrame): Input DataFrame with telemetry data (original raw data with Time index)
        - telemetry_metadata (dict): Dictionary containing telemetry metadata
        - window_weeks_list (list): List of time windows in weeks to analyze (default: [4, 8, 12])
        - rolling_window_minutes (int): Rolling mean window size in minutes (default: 30)
        
    Returns:
        - pd.DataFrame: DataFrame with trend analysis results for all unit-feature-window combinations
    '''
    
    # Get all features to analyze
    features = [signal['name'] for signal in telemetry_metadata['signals'] if signal.get('threshold_compute', False)]
    
    # Get all units
    if isinstance(df_in.index, pd.MultiIndex):
        units = df_in.index.get_level_values(0).unique()
    else:
        units = df_in[UNIT_COLNAME].unique()
    
    all_trends = []
    
    # Process each unit-feature-window combination
    for unit in units:
        for feature in features:
            for window_weeks in window_weeks_list:
                trend_result = analyze_trend_unit_feature(
                    df_in, 
                    unit, 
                    feature, 
                    window_weeks=window_weeks,
                    rolling_window_minutes=rolling_window_minutes
                )
                
                if trend_result is not None:
                    # Add risk direction information
                    feature_metadata = next((s for s in telemetry_metadata['signals'] if s['name'] == feature), None)
                    if feature_metadata:
                        trend_result['risk_direction'] = feature_metadata.get('risk_direction', 'unknown')
                        
                        # Interpret trend based on risk direction
                        slope_per_day = trend_result['slope_per_day']
                        risk_dir = trend_result['risk_direction']
                        
                        if risk_dir == 'high' and slope_per_day > 0:
                            trend_result['trend_interpretation'] = 'worsening'
                        elif risk_dir == 'low' and slope_per_day < 0:
                            trend_result['trend_interpretation'] = 'worsening'
                        elif risk_dir == 'both' and abs(slope_per_day) > 0:
                            trend_result['trend_interpretation'] = 'drifting'
                        else:
                            trend_result['trend_interpretation'] = 'improving'
                    
                    all_trends.append(trend_result)
    
    if all_trends:
        return pd.DataFrame(all_trends)
    else:
        return pd.DataFrame()

In [29]:
# Test Trend Analysis
trend_results = analyze_trends(df_aux, telemetry_metadata, window_weeks_list=[4, 8, 12])

# Quick overview - Filter for significant trends
significant_trends = trend_results[
    (trend_results['is_significant'] == True) & 
    (trend_results['is_good_fit'] == True)
]

print("📈 Trend Analysis Summary")
print(f"Total trends analyzed: {len(trend_results)}")
print(f"Statistically significant (p<{P_VALUE_THRESHOLD}): {trend_results['is_significant'].sum()}")
print(f"Good fit (R²>{R2_THRESHOLD}): {trend_results['is_good_fit'].sum()}")
print(f"Both significant & good fit: {len(significant_trends)}")

if len(significant_trends) > 0:
    print("\n⚠️ Top Concerning Trends (Significant + Good Fit + Worsening):")
    concerning = significant_trends[significant_trends['trend_interpretation'] == 'worsening'].copy()
    concerning['abs_slope'] = concerning['slope_per_day'].abs()
    concerning = concerning.sort_values('abs_slope', ascending=False).head(10)
    
    for _, row in concerning.iterrows():
        print(f"  {row['unit']} | {row['feature']} ({row['window_weeks']}w): "
              f"{row['slope_per_day']:+.3f}/day (R²={row['r2']:.2f}, p={row['p_value']:.4f})")

📈 Trend Analysis Summary
Total trends analyzed: 594
Statistically significant (p<0.05): 543
Good fit (R²>0.3): 6
Both significant & good fit: 6

⚠️ Top Concerning Trends (Significant + Good Fit + Worsening):
  T_13 | DiffLubePres (8w): -1.017/day (R²=0.34, p=0.0000)
  T_13 | DiffLubePres (12w): -1.017/day (R²=0.34, p=0.0000)
  T_17 | LckupSlip (8w): +0.031/day (R²=0.32, p=0.0000)


### Distribution Analysis

Then, we apply distribution shift analysis to detect significant changes in feature distributions over time.

#### Overview

We analyze whether features exhibit **statistically significant shifts in distribution** that may indicate progressive degradation, improvement, or drift in equipment performance. Unlike trend analysis (which looks at mean changes), this detects changes in the entire distribution shape.

#### Computation Details

**Time Window Strategy:**
We compare recent data against historical baseline using multiple lookback periods:
- **Observation Period** (recent): 4, 8, or 12 weeks
- **Baseline Period** (historical): 1 year of data (excluding observation period)

This creates three comparison scenarios:
- Last 4 weeks vs. previous year → detect recent shifts
- Last 8 weeks vs. previous year → detect medium-term shifts  
- Last 12 weeks vs. previous year → detect sustained shifts

**Statistical Method:**
- **Test**: Mann-Whitney U test (two-tailed, non-parametric)
- **Why Mann-Whitney**: Detects distribution shifts without assuming normality
- **State Control**: Analysis performed separately for each operational state (e.g., Operating, Idle, Hauling)
- **Outputs**:
  - **p-value**: Statistical significance of the distribution difference
  - **Effect size**: Magnitude of the shift (median difference or rank-biserial correlation)

**Scope:**
- Computed for every combination of:
  - Unit (individual equipment)
  - Feature (telemetry signal)
  - Operational State (Operating, Idle, etc.)
  - Observation window (4, 8, 12 weeks)

#### Implementation Approach

We implement the **Mann-Whitney U test** to determine if the recent data distribution has shifted compared to historical baseline, controlling for operational state to avoid confounding factors.

**Significance Criteria:**
- **Statistically significant**: p-value < 0.05 (95% confidence)
- **Practically significant**: Effect size exceeds threshold based on feature characteristics
  - Small effect: Cohen's d > 0.2
  - Medium effect: Cohen's d > 0.5
  - Large effect: Cohen's d > 0.8

**State Control Rationale:**
Analyzing each operational state separately ensures that distribution shifts reflect true changes in equipment behavior, not changes in operational patterns (e.g., more time in Idle vs. Operating).

#### Practical Application

Distribution analysis helps identify:
- 📊 **Behavior changes**: Equipment operating differently even if average values seem normal
- 📉 **Distribution shifts**: Increased variability or skewness indicating instability
- 🔄 **Mode changes**: Bimodal distributions suggesting intermittent issues
- ⚠️ **Early degradation**: Subtle distribution changes before mean trends become significant

**Example**: Engine oil pressure showing a significant distribution shift in the last 8 weeks (p=0.003) with lower median values, even though the rolling mean appears stable, suggests potential pump degradation or increased bearing clearances.

#### Output Interpretation

**Interpreting Results:**
- **Significant + Large Effect + High Risk Direction**: Recent values shifted higher → potential degradation
- **Significant + Large Effect + Low Risk Direction**: Recent values shifted lower → potential degradation  
- **Significant + Small Effect**: Statistically detectable but may not warrant immediate action
- **Non-significant**: Distribution remains consistent with historical baseline

In [30]:
# Distribution Shift Analysis Implementation
from scipy.stats import mannwhitneyu

BASELINE_WEEKS = 52  # 1 year baseline

def calculate_cohens_d(baseline, observation):
    '''Calculate Cohen's d effect size'''
    pooled_std = np.sqrt(((len(baseline) - 1) * baseline.std()**2 + (len(observation) - 1) * observation.std()**2) / (len(baseline) + len(observation) - 2))
    if pooled_std == 0:
        return 0
    return (observation.mean() - baseline.mean()) / pooled_std

def analyze_distribution_shift_unit_feature_state(
    df_in: pd.DataFrame,
    unit: str,
    feature: str,
    state: str,
    observation_weeks: int = 4,
    baseline_weeks: int = BASELINE_WEEKS,
    time_col: str = TIME_COLNAME
) -> dict:
    '''
    Analyze distribution shift for a specific unit-feature-state combination.
    
    Parameters:
        - df_in (pd.DataFrame): Input DataFrame with telemetry data
        - unit (str): The unit identifier to analyze
        - feature (str): The feature name to analyze
        - state (str): The operational state to analyze
        - observation_weeks (int): Recent observation period in weeks (default: 4)
        - baseline_weeks (int): Historical baseline period in weeks (default: 52)
        - time_col (str): The name of the time column
        
    Returns:
        - dict: Dictionary with distribution shift analysis results or None if insufficient data
    '''
    
    # Filter for specific unit and state
    if isinstance(df_in.index, pd.MultiIndex):
        unit_data = df_in.loc[unit].copy()
    else:
        unit_data = df_in[df_in[UNIT_COLNAME] == unit].copy()
    
    if feature not in unit_data.columns or STATE_COLNAME not in unit_data.columns:
        return None
    
    # Filter by state
    state_data = unit_data[unit_data[STATE_COLNAME] == state].copy()
    
    if len(state_data) == 0:
        return None
    
    # Sort by time
    state_data = state_data.sort_index() if isinstance(state_data.index, pd.DatetimeIndex) else state_data.sort_values(time_col)
    
    # Define time windows
    if isinstance(state_data.index, pd.DatetimeIndex):
        max_time = state_data.index.max()
    else:
        state_data = state_data.set_index(time_col)
        max_time = state_data.index.max()
    
    observation_start = max_time - pd.Timedelta(weeks=observation_weeks)
    baseline_end = observation_start
    baseline_start = max_time - pd.Timedelta(weeks=baseline_weeks)
    
    # Extract observation and baseline data
    observation_data = state_data[(state_data.index >= observation_start) & (state_data.index <= max_time)][feature].dropna()
    baseline_data = state_data[(state_data.index >= baseline_start) & (state_data.index < baseline_end)][feature].dropna()
    
    # Check minimum data requirements
    if len(observation_data) < 30 or len(baseline_data) < 100:  # Need sufficient samples
        return None
    
    # Perform Mann-Whitney U test
    try:
        u_statistic, p_value = mannwhitneyu(baseline_data, observation_data, alternative='two-sided')
    except Exception as e:
        return None
    
    # Calculate effect size (Cohen's d)
    cohens_d = calculate_cohens_d(baseline_data.values, observation_data.values)
    
    # Calculate medians
    baseline_median = baseline_data.median()
    observation_median = observation_data.median()
    median_diff = observation_median - baseline_median
    median_pct_change = (median_diff / baseline_median * 100) if baseline_median != 0 else 0
    
    return {
        'unit': unit,
        'feature': feature,
        'state': state,
        'observation_weeks': observation_weeks,
        'p_value': p_value,
        'u_statistic': u_statistic,
        'cohens_d': cohens_d,
        'effect_size_category': 'large' if abs(cohens_d) > 0.8 else ('medium' if abs(cohens_d) > 0.5 else ('small' if abs(cohens_d) > 0.2 else 'negligible')),
        'is_significant': p_value < 0.05,
        'baseline_median': baseline_median,
        'observation_median': observation_median,
        'median_diff': median_diff,
        'median_pct_change': median_pct_change,
        'baseline_n': len(baseline_data),
        'observation_n': len(observation_data),
        'baseline_start': baseline_start,
        'baseline_end': baseline_end,
        'observation_start': observation_start,
        'observation_end': max_time
    }


def analyze_distribution_shifts(
    df_in: pd.DataFrame,
    telemetry_metadata: dict,
    observation_weeks_list: list = [4, 8, 12],
    baseline_weeks: int = BASELINE_WEEKS
) -> pd.DataFrame:
    '''
    Analyze distribution shifts across all units, features, and states for multiple observation windows.
    
    Parameters:
        - df_in (pd.DataFrame): Input DataFrame with telemetry data (original raw data with Time index)
        - telemetry_metadata (dict): Dictionary containing telemetry metadata
        - observation_weeks_list (list): List of observation windows in weeks (default: [4, 8, 12])
        - baseline_weeks (int): Baseline period in weeks (default: 52 = 1 year)
        
    Returns:
        - pd.DataFrame: DataFrame with distribution shift analysis results
    '''
    
    # Get all features to analyze
    features = [signal['name'] for signal in telemetry_metadata['signals'] if signal.get('threshold_compute', False)]
    
    # Get all units
    if isinstance(df_in.index, pd.MultiIndex):
        units = df_in.index.get_level_values(0).unique()
    else:
        units = df_in[UNIT_COLNAME].unique()
    
    # Get all states
    if STATE_COLNAME in df_in.columns:
        states = df_in[STATE_COLNAME].unique()
    else:
        # If data is multi-indexed, reset to access state column
        temp_df = df_in.reset_index() if isinstance(df_in.index, pd.MultiIndex) else df_in
        states = temp_df[STATE_COLNAME].unique() if STATE_COLNAME in temp_df.columns else []
    
    all_shifts = []
    
    # Process each unit-feature-state-window combination
    for unit in units:
        for feature in features:
            for state in states:
                if pd.isna(state):  # Skip null states
                    continue
                for observation_weeks in observation_weeks_list:
                    shift_result = analyze_distribution_shift_unit_feature_state(
                        df_in,
                        unit,
                        feature,
                        state,
                        observation_weeks=observation_weeks,
                        baseline_weeks=baseline_weeks
                    )
                    
                    if shift_result is not None:
                        # Add risk direction information
                        feature_metadata = next((s for s in telemetry_metadata['signals'] if s['name'] == feature), None)
                        if feature_metadata:
                            shift_result['risk_direction'] = feature_metadata.get('risk_direction', 'unknown')
                            
                            # Interpret shift based on risk direction
                            median_diff = shift_result['median_diff']
                            risk_dir = shift_result['risk_direction']
                            
                            if risk_dir == 'high' and median_diff > 0:
                                shift_result['shift_interpretation'] = 'worsening'
                            elif risk_dir == 'low' and median_diff < 0:
                                shift_result['shift_interpretation'] = 'worsening'
                            elif risk_dir == 'both' and abs(median_diff) > 0:
                                shift_result['shift_interpretation'] = 'drifting'
                            else:
                                shift_result['shift_interpretation'] = 'improving'
                        
                        all_shifts.append(shift_result)
    
    if all_shifts:
        return pd.DataFrame(all_shifts)
    else:
        return pd.DataFrame()

In [31]:
# Test Distribution Shift Analysis
distribution_results = analyze_distribution_shifts(df_aux, telemetry_metadata, observation_weeks_list=[4, 8, 12])

# Filter for significant shifts with large effect
significant_shifts = distribution_results[
    (distribution_results['is_significant'] == True) & 
    (distribution_results['effect_size_category'].isin(['medium', 'large']))
]

print("📊 Distribution Shift Analysis Summary")
print(f"Total shifts analyzed: {len(distribution_results)}")
print(f"Statistically significant (p<0.05): {distribution_results['is_significant'].sum()}")
print(f"Medium or Large effect: {distribution_results['effect_size_category'].isin(['medium', 'large']).sum()}")
print(f"Both significant & meaningful effect: {len(significant_shifts)}")

if len(significant_shifts) > 0:
    print("\n⚠️ Top 10 Concerning Distribution Shifts (Significant + Large Effect + Worsening):")
    concerning = significant_shifts[significant_shifts['shift_interpretation'] == 'worsening'].copy()
    concerning['abs_cohens_d'] = concerning['cohens_d'].abs()
    concerning = concerning.sort_values('abs_cohens_d', ascending=False).head(10)
    
    for _, row in concerning.iterrows():
        print(f"  {row['unit']} | {row['feature']} | {row['state']} ({row['observation_weeks']}w): "
              f"Δ={row['median_diff']:+.2f} ({row['median_pct_change']:+.1f}%), "
              f"d={row['cohens_d']:.2f} ({row['effect_size_category']}), p={row['p_value']:.4f}")

📊 Distribution Shift Analysis Summary
Total shifts analyzed: 990
Statistically significant (p<0.05): 750
Medium or Large effect: 85
Both significant & meaningful effect: 85

⚠️ Top 10 Concerning Distribution Shifts (Significant + Large Effect + Worsening):
  T_11 | CnkcasePres | ND (8w): Δ=+0.38 (-107.1%), d=1.56 (large), p=0.0000
  T_11 | RtExhTemp | ND (4w): Δ=+94.91 (+115.2%), d=1.40 (large), p=0.0000
  T_11 | LtExhTemp | ND (4w): Δ=+81.97 (+101.7%), d=1.37 (large), p=0.0000
  T_11 | StrgOilTemp | ND (4w): Δ=+19.75 (+45.7%), d=1.36 (large), p=0.0000
  T_16 | CnkcasePres | ND (4w): Δ=+0.27 (-79.1%), d=1.23 (large), p=0.0000
  T_13 | DiffLubePres | Ralenti (4w): Δ=-51.31 (-42.2%), d=-1.19 (large), p=0.0000
  T_17 | CnkcasePres | ND (4w): Δ=+0.40 (-123.5%), d=1.16 (large), p=0.0000
  T_16 | CnkcasePres | ND (8w): Δ=+0.16 (-46.9%), d=1.01 (large), p=0.0000
  T_11 | TrnLubeTemp | ND (4w): Δ=+25.46 (+61.7%), d=0.99 (large), p=0.0000
  T_11 | RtFBrkTemp | ND (4w): Δ=+21.25 (+41.2%), d=0.98

### Anomaly Detection

Finally, we apply a **novel deep learning approach** to detect anomalies in telemetry features using an LSTM-based autoencoder architecture.

#### Overview

We analyze whether features exhibit **anomalies in production environment** that may indicate abnormal behavior in signals. Unlike threshold-based detection (Deviation Analysis), this approach learns complex temporal patterns and identifies deviations from normal operational signatures.

#### Computation Details

**Data Preparation:**
- **Window Size**: 30-minute sequences of telemetry data
- **Feature Grouping**: Signals organized by equipment system (defined in `signal_registry`)
  - Engine system signals
  - Hydraulic system signals
  - Transmission system signals
  - etc.
- **Missing Value Imputation**: Linear interpolation based on timestamps
- **Categorical Encoding**:
  - `Estado` (State) → One-Hot Encoding
  - `EngSpd` (Engine Speed) → Binned into 300 RPM intervals (<300, 300-600, 600-900, 900-1200, >1200)

**Data Quality Rules:**
- **Quality Threshold**: Only sequences with <10% imputed values are used for training and inference
- Ensures model learns from high-quality data patterns

**Autoencoder Architecture:**
- **Model Type**: LSTM (Long Short-Term Memory) Autoencoder
- **Input Layer**: 30-minute sequence × [Features per System + Encoded Categorical Features]
- **Encoder**: LSTM layers compress temporal patterns into latent representation
- **Decoder**: LSTM layers reconstruct original input from latent space
- **Output Layer**: Reconstructed sequence matching input dimensions

**Training Strategy:**
- **Training Data**: Sequences labeled as "normal" operation (from Deviation Analysis)
- **Objective**: Minimize reconstruction error for normal patterns
- **Validation**: Hold-out set of normal data to prevent overfitting

**Inference Pipeline:**
1. Feed new 30-minute sequence through trained autoencoder
2. Calculate reconstruction error (MSE between input and output)
3. Compare error against baseline distribution from training
4. Assign anomaly score (percentile rank or z-score)
5. Flag sequences exceeding threshold (e.g., >95th percentile)

**Scope:**
- Computed for every:
  - Unit (individual equipment)
  - System (group of related signals)
  - 30-minute rolling window

#### Implementation Approach

The autoencoder learns the **"normal operating signature"** of equipment systems during training. When applied to new data:

- **Low reconstruction error** → Sequence matches normal patterns → No anomaly
- **High reconstruction error** → Sequence deviates from learned patterns → Potential anomaly

**Why LSTM?**
- Captures temporal dependencies in sequential telemetry data
- Learns complex multi-signal correlations
- Detects subtle pattern deviations invisible to threshold-based methods

**Anomaly Score Interpretation:**
- **Score 0-70**: Normal operation
- **Score 70-90**: Minor deviation (monitor)
- **Score 90-95**: Moderate anomaly (investigate)
- **Score >95**: Severe anomaly (urgent attention)

#### Practical Application

Anomaly detection helps identify:
- 🔍 **Complex multi-signal anomalies**: Issues involving coordination between multiple systems
- 🎯 **Operational misuse**: Equipment operated outside normal patterns (e.g., rapid load changes, improper shutdown sequences)
- 🚨 **Early failure signatures**: Precursor patterns before component failures
- 📊 **Intermittent issues**: Anomalies that don't trigger static thresholds but represent abnormal behavior

**Advantages over Threshold Methods:**
- Detects **pattern anomalies** not just value anomalies
- Learns **system-level correlations** across multiple signals
- Adapts to **normal variability** in operating conditions
- Identifies **novel failure modes** not seen in historical data

**Example**: An autoencoder detects high reconstruction error for a 30-minute window where engine coolant temperature, oil pressure, and RPM variations show unusual coordination—individually within normal ranges but collectively indicating operator misuse (rapid acceleration cycles) or early water pump bearing issues.

#### Model Lifecycle

**Training Phase:**
1. Collect 3-6 months of "normal" operation data
2. Preprocess and create 30-minute sequences
3. Train LSTM autoencoder to minimize reconstruction error
4. Validate on hold-out normal data
5. Establish baseline error distribution

**Production Phase:**
1. Process incoming data in 30-minute rolling windows
2. Apply trained model to compute reconstruction error
3. Calculate anomaly score against baseline
4. Flag high-score sequences for operator/analyst review
5. Periodically retrain model with new normal data

**Continuous Improvement:**
- Anomaly scores reviewed by domain experts
- Confirmed normal patterns added to training set (reduce false positives)
- Confirmed anomalies archived for failure mode analysis

In [40]:
# LSTM Autoencoder Anomaly Detection Implementation
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

SEQUENCE_LENGTH = 30  # 30-minute sequences
QUALITY_THRESHOLD = 0.10  # <10% imputed values
ENG_SPD_BINS = [0, 300, 600, 900, 1200, float('inf')]  # RPM bins
ENG_SPD_LABELS = ['<300', '300-600', '600-900', '900-1200', '>1200']


def get_system_features(telemetry_metadata: dict, system_name: str) -> list:
    '''Get all features belonging to a specific system.'''
    return [signal['name'] for signal in telemetry_metadata['signals'] 
            if signal.get('system', '') == system_name and signal.get('threshold_compute', False)]


def encode_categorical_features(df_in: pd.DataFrame) -> pd.DataFrame:
    '''
    Encode categorical features for LSTM input.
    - Estado (State): One-Hot Encoding
    - EngSpd (Engine Speed): Binned into 300 RPM intervals
    
    Parameters:
        - df_in (pd.DataFrame): Input DataFrame
        
    Returns:
        - pd.DataFrame: DataFrame with encoded features
    '''
    df_encoded = df_in.copy()
    
    # One-hot encode Estado (State)
    if STATE_COLNAME in df_encoded.columns:
        estado_dummies = pd.get_dummies(df_encoded[STATE_COLNAME], prefix='Estado')
        df_encoded = pd.concat([df_encoded, estado_dummies], axis=1)
    
    # Bin EngSpd if it exists
    if 'EngSpd' in df_encoded.columns:
        df_encoded['EngSpd_binned'] = pd.cut(
            df_encoded['EngSpd'], 
            bins=ENG_SPD_BINS, 
            labels=ENG_SPD_LABELS,
            include_lowest=True
        )
        engspd_dummies = pd.get_dummies(df_encoded['EngSpd_binned'], prefix='EngSpd_bin')
        df_encoded = pd.concat([df_encoded, engspd_dummies], axis=1)
        # Drop the categorical binned column to avoid string values
        df_encoded = df_encoded.drop(columns=['EngSpd_binned'])
    
    return df_encoded


def prepare_sequences(
    df_in: pd.DataFrame,
    unit: str,
    system_features: list,
    sequence_length: int = SEQUENCE_LENGTH,
    quality_threshold: float = QUALITY_THRESHOLD
) -> tuple:
    '''
    Prepare 30-minute sequences with quality checks.
    
    Parameters:
        - df_in (pd.DataFrame): Input DataFrame with telemetry data
        - unit (str): Unit identifier
        - system_features (list): List of features in the system
        - sequence_length (int): Sequence length in minutes (default: 30)
        - quality_threshold (float): Maximum allowed imputation ratio (default: 0.10)
        
    Returns:
        - tuple: (sequences, quality_flags) or (None, None) if insufficient data
    '''
    
    # Filter for specific unit
    if isinstance(df_in.index, pd.MultiIndex):
        unit_data = df_in.loc[unit].copy()
    else:
        unit_data = df_in[df_in[UNIT_COLNAME] == unit].copy()
    
    # Sort by time
    unit_data = unit_data.sort_index() if isinstance(unit_data.index, pd.DatetimeIndex) else unit_data.sort_values(TIME_COLNAME).set_index(TIME_COLNAME)
    
    # Encode categorical features
    unit_data = encode_categorical_features(unit_data)
    
    # Get encoded feature columns
    encoded_estado_cols = [col for col in unit_data.columns if col.startswith('Estado_')]
    encoded_engspd_cols = [col for col in unit_data.columns if col.startswith('EngSpd_bin_')]
    
    # Select features for the system + encoded categoricals
    numerical_cols = [f for f in system_features if f in unit_data.columns]
    feature_cols = numerical_cols + encoded_estado_cols + encoded_engspd_cols
    
    if len(feature_cols) == 0:
        return None, None
    
    # Track missing values before imputation
    missing_mask = unit_data[feature_cols].isna()
    
    # Linear interpolation for numerical features
    if len(numerical_cols) > 0:
        unit_data[numerical_cols] = unit_data[numerical_cols].interpolate(method='time', limit_direction='both')
    
    # Forward fill for categorical encoded features (one-hot and binned)
    categorical_cols = encoded_estado_cols + encoded_engspd_cols
    if len(categorical_cols) > 0:
        unit_data[categorical_cols] = unit_data[categorical_cols].ffill().bfill()
    
    # Create sequences
    sequences = []
    quality_flags = []
    
    for i in range(len(unit_data) - sequence_length + 1):
        sequence = unit_data[feature_cols].iloc[i:i + sequence_length].values
        
        # Calculate imputation ratio for this sequence
        missing_count = missing_mask[feature_cols].iloc[i:i + sequence_length].sum().sum()
        total_values = sequence_length * len(feature_cols)
        imputation_ratio = missing_count / total_values if total_values > 0 else 1.0
        
        # Only include high-quality sequences
        if imputation_ratio < quality_threshold:
            sequences.append(sequence)
            quality_flags.append(imputation_ratio)
    
    if len(sequences) == 0:
        return None, None
    
    return np.array(sequences), np.array(quality_flags)


def build_lstm_autoencoder(input_shape: tuple, encoding_dim: int = 32) -> keras.Model:
    '''
    Build LSTM Autoencoder model.
    
    Parameters:
        - input_shape (tuple): (sequence_length, n_features)
        - encoding_dim (int): Dimension of latent representation (default: 32)
        
    Returns:
        - keras.Model: Compiled LSTM autoencoder model
    '''
    
    # Encoder
    encoder_inputs = keras.Input(shape=input_shape)
    x = layers.LSTM(64, activation='relu', return_sequences=True)(encoder_inputs)
    x = layers.LSTM(encoding_dim, activation='relu', return_sequences=False)(x)
    encoder = keras.Model(encoder_inputs, x, name='encoder')
    
    # Decoder
    decoder_inputs = keras.Input(shape=(encoding_dim,))
    x = layers.RepeatVector(input_shape[0])(decoder_inputs)
    x = layers.LSTM(encoding_dim, activation='relu', return_sequences=True)(x)
    x = layers.LSTM(64, activation='relu', return_sequences=True)(x)
    decoder_outputs = layers.TimeDistributed(layers.Dense(input_shape[1]))(x)
    decoder = keras.Model(decoder_inputs, decoder_outputs, name='decoder')
    
    # Autoencoder
    autoencoder_inputs = keras.Input(shape=input_shape)
    encoded = encoder(autoencoder_inputs)
    decoded = decoder(encoded)
    autoencoder = keras.Model(autoencoder_inputs, decoded, name='autoencoder')
    
    # Compile
    autoencoder.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse'
    )
    
    return autoencoder


def train_autoencoder_unit_system(
    df_in: pd.DataFrame,
    deviation_results: pd.DataFrame,
    unit: str,
    system_name: str,
    system_features: list,
    epochs: int = 50,
    batch_size: int = 32,
    validation_split: float = 0.2
) -> dict:
    '''
    Train LSTM autoencoder for a specific unit and system.
    
    Parameters:
        - df_in (pd.DataFrame): Input DataFrame with telemetry data
        - deviation_results (pd.DataFrame): Deviation analysis results (to identify normal data)
        - unit (str): Unit identifier
        - system_name (str): System name
        - system_features (list): Features in the system
        - epochs (int): Training epochs (default: 50)
        - batch_size (int): Batch size (default: 32)
        - validation_split (float): Validation split ratio (default: 0.2)
        
    Returns:
        - dict: Training results including model, scaler, and metrics
    '''
    
    # Filter for normal sequences based on deviation analysis
    # A sequence is "normal" if all features in the system are marked as normal
    if isinstance(deviation_results.index, pd.MultiIndex):
        unit_deviations = deviation_results.loc[unit].copy()
    else:
        unit_deviations = deviation_results[deviation_results[UNIT_COLNAME] == unit].copy()
    
    # Check if all system features are normal
    risk_cols = [f'risk_level_{f}' for f in system_features if f'risk_level_{f}' in unit_deviations.columns]
    
    if len(risk_cols) == 0:
        return None
    
    # Filter normal data (all features normal)
    normal_mask = (unit_deviations[risk_cols] == 'normal').all(axis=1)
    
    # Get unit data from df_in
    if isinstance(df_in.index, pd.MultiIndex):
        unit_raw_data = df_in.loc[unit].copy()
    else:
        unit_raw_data = df_in[df_in[UNIT_COLNAME] == unit].copy()
        if TIME_COLNAME in unit_raw_data.columns:
            unit_raw_data = unit_raw_data.set_index(TIME_COLNAME)
    
    # Align indices and filter normal data - keep only timestamps where all risk levels are normal
    normal_timestamps = normal_mask[normal_mask].index
    normal_data = unit_raw_data[unit_raw_data.index.isin(normal_timestamps)]
    
    # Prepare sequences
    sequences, quality_flags = prepare_sequences(normal_data, unit, system_features)
    
    if sequences is None or len(sequences) < 100:  # Need minimum sequences
        return None
    
    # Normalize features
    scaler = StandardScaler()
    n_samples, n_timesteps, n_features = sequences.shape
    sequences_flat = sequences.reshape(-1, n_features)
    sequences_scaled = scaler.fit_transform(sequences_flat).reshape(n_samples, n_timesteps, n_features)
    
    # Split train/validation
    X_train, X_val = train_test_split(sequences_scaled, test_size=validation_split, random_state=42)
    
    # Build model
    input_shape = (n_timesteps, n_features)
    model = build_lstm_autoencoder(input_shape)
    
    # Early stopping
    early_stopping = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )
    
    # Train
    history = model.fit(
        X_train, X_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_val, X_val),
        callbacks=[early_stopping],
        verbose=0
    )
    
    # Calculate baseline reconstruction errors
    train_reconstructions = model.predict(X_train, verbose=0)
    train_mse = np.mean(np.square(X_train - train_reconstructions), axis=(1, 2))
    
    val_reconstructions = model.predict(X_val, verbose=0)
    val_mse = np.mean(np.square(X_val - val_reconstructions), axis=(1, 2))
    
    # Establish baseline statistics
    baseline_mean = np.mean(train_mse)
    baseline_std = np.std(train_mse)
    baseline_p95 = np.percentile(train_mse, 95)
    baseline_p99 = np.percentile(train_mse, 99)
    
    return {
        'unit': unit,
        'system': system_name,
        'model': model,
        'scaler': scaler,
        'n_features': n_features,
        'n_sequences': len(sequences),
        'train_loss': history.history['loss'][-1],
        'val_loss': history.history['val_loss'][-1],
        'baseline_mean': baseline_mean,
        'baseline_std': baseline_std,
        'baseline_p95': baseline_p95,
        'baseline_p99': baseline_p99,
        'feature_columns': system_features
    }


def train_autoencoders(
    df_in: pd.DataFrame,
    deviation_results: pd.DataFrame,
    telemetry_metadata: dict,
    epochs: int = 50
) -> dict:
    '''
    Train LSTM autoencoders for all unit-system combinations.
    
    Parameters:
        - df_in (pd.DataFrame): Input DataFrame with telemetry data
        - deviation_results (pd.DataFrame): Deviation analysis results
        - telemetry_metadata (dict): Telemetry metadata with system definitions
        - epochs (int): Training epochs per model (default: 50)
        
    Returns:
        - dict: Dictionary of trained models keyed by (unit, system)
    '''
    
    # Get all systems
    systems = set([signal.get('system', 'default') for signal in telemetry_metadata['signals'] if signal.get('threshold_compute', False)])
    systems = {s for s in systems if s and s != 'default'}
    
    # Get all units
    if isinstance(df_in.index, pd.MultiIndex):
        units = df_in.index.get_level_values(0).unique()
    else:
        units = df_in[UNIT_COLNAME].unique()
    
    trained_models = {}
    
    print(f"Training autoencoders for {len(units)} units × {len(systems)} systems...")
    
    for unit in units:
        for system_name in systems:
            system_features = get_system_features(telemetry_metadata, system_name)
            
            if len(system_features) == 0:
                continue
            
            print(f"  Training: {unit} | {system_name} ({len(system_features)} features)...", end='')
            
            result = train_autoencoder_unit_system(
                df_in,
                deviation_results,
                unit,
                system_name,
                system_features,
                epochs=epochs
            )
            
            if result is not None:
                trained_models[(unit, system_name)] = result
                print(f" ✓ ({result['n_sequences']} sequences, val_loss={result['val_loss']:.4f})")
            else:
                print(" ✗ (insufficient data)")
    
    print(f"\nTraining complete: {len(trained_models)} models trained successfully.")
    
    return trained_models

In [41]:
# Test Autoencoder Training
# Note: This requires system definitions in signal_registry
# Example: signals should have a 'system' field like 'engine', 'hydraulic', 'transmission'

print("🤖 LSTM Autoencoder Training")
print("="*60)
print("Prerequisites:")
print("  - signal_registry.yaml must have 'system' field for each signal")
print("  - Example: system: 'engine' for EngCoolTemp, EngOilPress, etc.")
print("  - Training uses normal sequences from deviation analysis (r)")
print("\n⚠️  Training may take several minutes depending on data size...")
print("="*60)

# Train autoencoders (uncomment to run)
trained_models = train_autoencoders(df_aux, r, telemetry_metadata, epochs=1)

# Summary (after training)
print(f"\n📊 Training Summary:")
print(f"Total models trained: {len(trained_models)}")
for (unit, system), model_info in list(trained_models.items())[:5]:
    print(f"  {unit} | {system}: {model_info['n_sequences']} seq, "
          f"val_loss={model_info['val_loss']:.4f}, "
          f"baseline_p95={model_info['baseline_p95']:.4f}")

🤖 LSTM Autoencoder Training
Prerequisites:
  - signal_registry.yaml must have 'system' field for each signal
  - Example: system: 'engine' for EngCoolTemp, EngOilPress, etc.
  - Training uses normal sequences from deviation analysis (r)

⚠️  Training may take several minutes depending on data size...
Training autoencoders for 10 units × 4 systems...
  Training: T_14 | Transmission (5 features)...WARNING:tensorflow:TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.
 ✓ (50788 sequences, val_loss=52893.0117)
  Training: T_14 | Steering (1 features)... ✓ (91090 sequences, val_loss=4.8529)
  Training: T_14 | Brakes (4 features)... ✓ (81886 sequences, val_loss=66.1272)
  Training: T_14 | Engine (12 features)... ✗ (insufficient data)
  Training: T_17 | Transmission (5 features)... ✓ (43810 sequences, val_loss=0.4872)
  Training: T_17 | Steering (1 features)

: 